# Intro 08 — Law of Large Numbers and Central Limit Theorem

Practice notebook for the **"Law of Large Numbers and Central Limit Theorem"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
# Part 1 — Law of Large Numbers: the Sample Mean Converges

Let \(X_1, X_2, \dots\) be i.i.d. random variables with \(E[X_i] = \mu\). The **sample mean** of the first \(n\) observations is

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i.$$

The **law of large numbers (LLN)** states that, as \(n \to \infty\),

$$\bar{X}_n \to \mu \quad \text{(in probability)}.$$

In words: the sample mean converges to the true mean as we collect more data, and the size of typical fluctuations shrinks.

**PDF example.** A fair die has \(\mu = (1+2+3+4+5+6)/6 = 3.5\). As we roll more and more times, the running average creeps toward 3.5.


In [ ]:
# Fair die: true mean is 3.5
mu_die = 3.5
rng = np.random.default_rng(42)

# Simulate a long sequence of rolls and track the running average
n_rolls = 10_000
rolls = rng.integers(1, 7, size=n_rolls)          # 1..6 inclusive
running_mean = np.cumsum(rolls, dtype=float) / np.arange(1, n_rolls + 1)

print(f"First 5 rolls            : {rolls[:5].tolist()}")
print(f"Running mean after    1  : {running_mean[0]:.4f}")
print(f"Running mean after   10  : {running_mean[9]:.4f}")
print(f"Running mean after  100  : {running_mean[99]:.4f}")
print(f"Running mean after 1000  : {running_mean[999]:.4f}")
print(f"Running mean after 10000  : {running_mean[-1]:.4f}")
print(f"True mean                : {mu_die}")

fig, ax = plt.subplots()
ax.plot(np.arange(1, n_rolls + 1), running_mean, lw=1.0, label=r"$\bar{X}_n$")
ax.axhline(mu_die, color="tab:red", ls="--", label=r"$\mu = 3.5$")
ax.set_xscale("log")
ax.set_xlabel("number of rolls  $n$")
ax.set_ylabel(r"running mean  $\bar{X}_n$")
ax.set_title("Law of Large Numbers — fair die running average")
ax.legend()
plt.show()


**Your turn:** Repeat the experiment with a *biased* die that rolls 1..6 with probabilities `p = [0.1, 0.1, 0.1, 0.1, 0.1, 0.5]` (use `rng.choice(6, size=n_rolls, p=p) + 1`). What is the new true mean? Does the running average still converge to it?


---
# Part 2 — LLN with Coins: Shrinking Spread

Flip a fair coin and code heads as 1, tails as 0. Then \(X_i \sim \text{Bernoulli}(0.5)\) with \(\mu = 0.5\) and \(\sigma^2 = 0.25\). The LLN says the running fraction of heads converges to 0.5. Plotting several independent sequences shows the *spread* around \(\mu\) shrinking like \(1/\sqrt{n}\).


In [ ]:
rng = np.random.default_rng(7)
n_flips = 5_000
n_paths = 20
mu_coin = 0.5

# Matrix: each row is an independent sequence of coin flips
flips = rng.integers(0, 2, size=(n_paths, n_flips))
running = np.cumsum(flips, axis=1, dtype=float) / np.arange(1, n_flips + 1)

fig, ax = plt.subplots()
for i in range(n_paths):
    ax.plot(np.arange(1, n_flips + 1), running[i], lw=0.7, alpha=0.7)
ax.axhline(mu_coin, color="black", ls="--", lw=1.5, label=r"$\mu = 0.5$")
# Theoretical 1/sqrt(n) envelope: std of running mean is sigma/sqrt(n)
n_grid = np.arange(1, n_flips + 1)
sigma = 0.5
ax.fill_between(n_grid, mu_coin - 2*sigma/np.sqrt(n_grid), mu_coin + 2*sigma/np.sqrt(n_grid),
                color="tab:gray", alpha=0.2, label=r"$\mu \pm 2\sigma/\sqrt{n}$")
ax.set_xscale("log")
ax.set_xlabel("number of flips  $n$")
ax.set_ylabel(r"running fraction of heads")
ax.set_title("LLN — 20 coin-flip sequences converging to 0.5")
ax.legend()
plt.show()

# How wide is the spread at a few chosen n?
for n in [10, 100, 1000, 5000]:
    print(f"n={n:5d}: std of running mean across paths ~ {running[:, n-1].std():.4f}  "
          f"(theory {sigma/np.sqrt(n):.4f})")


**Your turn:** Replace `rng.integers(0, 2, ...)` with a *biased* coin `rng.choice(2, p=[0.7, 0.3])`. How do the convergence level and the \(1/\sqrt{n}\) envelope change?


---
# Part 3 — Central Limit Theorem: Sample Means Become Normal

Under mild conditions, the **central limit theorem (CLT)** says

$$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \Rightarrow N(0,1),$$

where \(\sigma^2 = \operatorname{Var}(X_i)\) and \(\Rightarrow\) is convergence in distribution. Equivalently, for large \(n\),

$$\bar{X}_n \approx N\!\left(\mu,\, \frac{\sigma^2}{n}\right).$$

**Intuition.** No matter the shape of the original distribution of \(X_i\), the average of many independent copies looks increasingly normal. We demonstrate this with an **exponential** distribution (strongly non-normal, skewed) and compare the histogram of sample means to the CLT-predicted normal limit \(N(\mu, \sigma^2/n)\).


In [ ]:
# Exponential(1): heavily right-skewed, mean=1, variance=1
rng = np.random.default_rng(123)
mu_exp, sigma_exp = 1.0, 1.0
n_sim = 50_000                   # number of sample means to collect

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, n in zip(axes, [1, 5, 30]):
    # Each row is a sample of size n; take the mean across columns
    draws = rng.exponential(scale=1.0, size=(n_sim, n))
    sample_means = draws.mean(axis=1)

    # CLT-predicted normal: N(mu, sigma^2 / n)
    clt_mean, clt_std = mu_exp, sigma_exp / np.sqrt(n)
    xs = np.linspace(sample_means.min(), sample_means.max(), 400)
    pdf = stats.norm.pdf(xs, loc=clt_mean, scale=clt_std)

    ax.hist(sample_means, bins=60, density=True, color="tab:steelblue", alpha=0.7)
    ax.plot(xs, pdf, color="tab:red", lw=2, label=r"$N(\mu,\,\sigma^2/n)$")
    ax.set_title(f"n = {n}")
    ax.set_xlabel(r"$\bar{X}_n$")
    if n == 1:
        ax.set_ylabel("density")
    ax.legend()

fig.suptitle("CLT — distribution of $\\bar{X}_n$ from Exponential(1) approaches normal", y=1.02)
plt.tight_layout()
plt.show()

# Numerical check: at n=30, is the standardized sample mean close to standard normal?
z30 = (sample_means - mu_exp) / (sigma_exp / np.sqrt(30))
print(f"n=30: empirical mean of z = {z30.mean():.4f}, std of z = {z30.std():.4f}")
print(f"      P(|z| <= 1.96) empirical = {np.mean(np.abs(z30) <= 1.96):.4f}  (theory 0.95)")


**Your turn:** Re-run the cell with a **uniform** distribution (`rng.uniform(0, 1, size=...)`, which has mean 0.5 and variance 1/12). How fast does the histogram approach the normal limit compared to the exponential case?


---
# Part 4 — CLT Is About the Distribution of \(\bar{X}_n\)

A common confusion: the CLT does **not** say the raw data \(X_i\) become normal — only the *sample mean* \(\bar{X}_n\) does. Below we overlay the raw data distribution and the distribution of the sample mean for a very skewed Bernoulli parent with \(p=0.1\).

Parent: \(X_i \sim \text{Bernoulli}(0.1)\) with \(\mu = 0.1\), \(\sigma^2 = 0.09\).


In [ ]:
rng = np.random.default_rng(99)
p = 0.1
mu_b, var_b = p, p*(1-p)
n_sim = 40_000
n = 50

raw = rng.choice([0, 1], size=n_sim, p=[1-p, p])                       # raw X_i
means = rng.choice([0, 1], size=(n_sim, n), p=[1-p, p]).mean(axis=1)  # Xbar_n

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(raw, bins=[-0.5, 0.5, 1.5], density=True, color="tab:gray", alpha=0.8, align="mid")
axes[0].set_title("Raw data  $X_i$  (Bernoulli(0.1)) — NOT normal")
axes[0].set_xlabel("value"); axes[0].set_ylabel("density")

# Distribution of the sample mean, plus CLT normal overlay
clt_std = np.sqrt(var_b / n)
xs = np.linspace(means.min(), means.max(), 400)
axes[1].hist(means, bins=50, density=True, color="tab:steelblue", alpha=0.7)
axes[1].plot(xs, stats.norm.pdf(xs, loc=mu_b, scale=clt_std), color="tab:red", lw=2,
            label=r"$N(\mu,\,\sigma^2/n)$")
axes[1].set_title(f"Distribution of $\\bar{{X}}_{{n={n}}}$ — already close to normal")
axes[1].set_xlabel(r"$\bar{X}_n$"); axes[1].legend()

plt.tight_layout(); plt.show()

print(f"Raw data    : mean={raw.mean():.4f} (theory {mu_b}), var={raw.var():.4f} (theory {var_b})")
print(f"Sample means: mean={means.mean():.4f}, var={means.var():.6f} (theory {var_b/n:.6f})")


**Your turn:** Set \(p=0.01\) and \(n=10\). The parent is almost all zeros — extremely skewed. Does the CLT normal approximation still work for \(\bar{X}_n\)? Why or why not? What happens if you increase \(n\) to 200?


---
# Part 5 — QQ-Plot: Visual Check Against the Normal Limit

A **quantile-quantile (QQ) plot** compares the quantiles of a sample to the quantiles of a theoretical distribution. If the sample comes from that distribution, points fall on a straight line. We standardize \(\bar{X}_n\) to

$$Z_n = \frac{\bar{X}_n - \mu}{\sigma/\sqrt{n}}$$

and QQ-plot it against \(N(0,1)\). Straight line ⇒ the CLT approximation is good.


In [ ]:
rng = np.random.default_rng(2024)
n = 50
n_sim = 20_000

# Three parents: normal (control), uniform, exponential
configs = [
    ("Normal(0,1)",     lambda: rng.standard_normal((n_sim, n)),        0.0, 1.0),
    ("Uniform(0,1)",    lambda: rng.uniform(0, 1, size=(n_sim, n)),     0.5, np.sqrt(1/12)),
    ("Exponential(1)", lambda: rng.exponential(1.0, size=(n_sim, n)), 1.0, 1.0),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, sampler, mu, sigma) in zip(axes, configs):
    means = sampler().mean(axis=1)
    z = (means - mu) / (sigma / np.sqrt(n))
    stats.probplot(z, dist="norm", plot=ax)
    ax.get_lines()[0].set_color("tab:steelblue")
    ax.get_lines()[0].set_markersize(2.5)
    ax.get_lines()[1].set_color("tab:red")
    ax.set_title(f"{name}\n(n={n})")
    ax.set_xlabel("theoretical normal quantiles")
fig.suptitle("QQ-plots of standardized $\\bar{X}_n$ against $N(0,1)$ — CLT in action", y=1.03)
plt.tight_layout(); plt.show()

# Tail check: empirical vs theoretical 1%/99% quantiles of Z_n
for name, sampler, mu, sigma in configs:
    means = sampler().mean(axis=1)
    z = (means - mu) / (sigma / np.sqrt(n))
    emp1, emp99 = np.percentile(z, [1, 99])
    th1, th99 = stats.norm.ppf([0.01, 0.99])
    print(f"{name:17s}: empirical 1%={emp1:.3f}, 99%={emp99:.3f}  |  "
          f"theory 1%={th1:.3f}, 99%={th99:.3f}")


**Your turn:** Reduce `n` from 50 to 5 and rerun. Which parent's QQ-plot deviates most from the straight line? Why does the exponential deviate more than the uniform? Restore `n=50` afterward.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the **law of large numbers** in your own words. If \(X_i\) are i.i.d. with \(E[X_i]=\mu\), what does \(\bar{X}_n\) converge to, and in what sense? Give one real-world example where you would expect the LLN to apply.

2. A fair six-sided die has \(\mu = 3.5\) and \(\sigma^2 = 35/12 \approx 2.9167\). Simulate one sequence of 100,000 rolls with a fixed seed. Report the running mean at \(n = 10, 100, 1000, 10000, 100000\). Confirm the gap \(|\bar{X}_n - \mu|\) shrinks roughly like \(1/\sqrt{n}\).

3. Let \(X_i \sim \text{Exponential}(\lambda=2)\), which has \(\mu = 1/2\) and \(\sigma^2 = 1/4\). Using the CLT, write the approximate distribution of \(\bar{X}_n\) for large \(n\). Then simulate 30,000 sample means with \(n=40\), overlay the predicted normal density, and report the empirical vs theoretical probability that \(\bar{X}_n \in [\mu - 1.96\sigma/\sqrt{n},\, \mu + 1.96\sigma/\sqrt{n}]\).

4. Explain why the CLT does **not** imply the raw observations \(X_i\) are normal. Construct a concrete counterexample with NumPy (e.g. a Bernoulli or exponential parent), plotting the raw data histogram next to the histogram of \(\bar{X}_n\) for \(n=50\).

5. Using `scipy.stats.probplot`, produce QQ-plots of the standardized sample mean of a **uniform(0,1)** parent against \(N(0,1)\) for \(n = 2, 10, 50\). Describe how the shape of the QQ-plot changes with \(n\) and explain what it tells you about how fast the CLT kicks in for this symmetric parent.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** The LLN says the sample mean \(\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i\) converges **in probability** to the true mean \(\mu = E[X_i]\): for any \(\varepsilon > 0\), \(P(|\bar{X}_n - \mu| > \varepsilon) \to 0\) as \(n \to \infty\). Example: the long-run average return of a repeated trading strategy converges to its expected single-period return.

**2.**

```python
import numpy as np
rng = np.random.default_rng(0)
rolls = rng.integers(1, 7, size=100_000)
cum = np.cumsum(rolls, dtype=float) / np.arange(1, 100_001)
mu, sigma2 = 3.5, 35/12
for n in [10, 100, 1000, 10000, 100000]:
    gap = abs(cum[n-1] - mu)
    print(f"n={n:6d}: Xbar={cum[n-1]:.5f}, gap={gap:.4f}, ~sigma/sqrt(n)={np.sqrt(sigma2/n):.4f}")
```
The gap at each \(n\) is of the same order as \(\sigma/\sqrt{n}\), confirming the \(1/\sqrt{n}\) shrinkage.

**3.** CLT gives \(\bar{X}_n \approx N(\mu = 0.5,\, \sigma^2/n = 0.25/40 = 0.00625)\). So \(\bar{X}_n \in [\mu - 1.96\sigma/\sqrt{n},\, \mu + 1.96\sigma/\sqrt{n}]\) with probability \(\approx 0.95\).

```python
import numpy as np
from scipy import stats
rng = np.random.default_rng(1)
mu, lam = 0.5, 2.0
n = 40
means = rng.exponential(1/lam, size=(30_000, n)).mean(axis=1)
se = np.sqrt(0.25/n)
lo, hi = mu - 1.96*se, mu + 1.96*se
print(f"empirical coverage = {np.mean((means >= lo) & (means <= hi)):.4f}  (theory 0.95)")
xs = np.linspace(means.min(), means.max(), 400)
import matplotlib.pyplot as plt
plt.hist(means, bins=60, density=True, alpha=0.7)
plt.plot(xs, stats.norm.pdf(xs, mu, se), "r-", lw=2); plt.show()
```
Empirical coverage should be close to 0.95.

**4.** The CLT concerns the *distribution of the average*, not the data. With a Bernoulli(0.1) parent the raw data are two spikes at 0 and 1 — clearly not normal — yet \(\bar{X}_{50}\) is already bell-shaped:

```python
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(2)
n = 50
raw = rng.choice([0,1], p=[0.9,0.1], size=20000)
means = rng.choice([0,1], p=[0.9,0.1], size=(20000, n)).mean(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].hist(raw, bins=[-0.5,0.5,1.5], density=True, align="mid", color="gray"); ax[0].set_title("Raw X_i")
ax[1].hist(means, bins=40, density=True, color="steelblue", alpha=0.7); ax[1].set_title("Xbar_n, n=50")
plt.show()
```

**5.**

```python
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(3)
fig, ax = plt.subplots(1, 3, figsize=(13,4))
for k, n in enumerate([2, 10, 50]):
    means = rng.uniform(0, 1, size=(20000, n)).mean(axis=1)
    z = (means - 0.5) / np.sqrt((1/12)/n)
    stats.probplot(z, dist="norm", plot=ax[k])
    ax[k].set_title(f"n={n}")
plt.tight_layout(); plt.show()
```
At \(n=2\) the plot shows mild S-shape (heavy-tailed relative to normal because the average of just two uniforms has a triangular distribution). By \(n=10\) it is nearly straight, and at \(n=50\) essentially a straight line — for a symmetric parent like uniform(0,1) the CLT converges quickly.

</details>
